# 第2回　代表値の性質と、その嘘
## ―― 平均・中央値・最頻値、そして外れ値

統計学Ⅰ（B）　／　北星学園大学

今日も**▶を上から押すだけ**でよい（コードを書くのは第4回から）。注目するのは一つ ――

> 同じデータでも、**どの代表値を使うかで「物語」が変わる**。

In [ ]:
# 準備：ライブラリと、北辰大学の学生データを読み込む。▶ を押すだけ。
!pip install -q japanize-matplotlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import japanize_matplotlib  # noqa: F401

URL = "https://aonoa68.github.io/toukei-1/data/hokushin_students.csv"
try:
    df = pd.read_csv(URL)          # 公開後はネットから読める
except Exception:
    # まだ公開前/オフラインのときは、同じデータをその場で作る（中身は気にしなくてよい）
    R = np.random.default_rng(2026); N = 400
    disc = R.normal(0,1,N); apt = R.normal(0,1,N)
    gk = R.choice(["経済学部","文学部","社会福祉学部"], N, p=[.40,.35,.25])
    gke = np.select([gk=="経済学部",gk=="文学部",gk=="社会福祉学部"],[2.,-1.,-1.])
    gen = R.choice(["女","男","回答しない"], N, p=[.55,.43,.02])
    bh = np.where(gen=="男",171.,158.); bh=np.where(gen=="回答しない",165.,bh)
    height = bh + R.normal(0,6,N)
    alone = R.random(N)<.35
    com = np.clip(np.where(alone,R.normal(20,8,N),R.normal(55,25,N)),5,None)
    slp = 7+.5*disc-.005*(com-30)+R.normal(0,.8,N)
    sns = np.clip(3-.8*disc+R.normal(0,1.,N),0,None)
    std = np.clip(1.5+.7*disc+.2*apt+R.normal(0,.6,N),0,None)
    pt  = np.clip(np.where(alone,R.normal(18,6,N),R.normal(10,6,N)),0,None)
    att = np.clip(82+7*disc+R.normal(0,5,N),0,100)
    bf  = np.clip(np.round(4+1.6*disc+R.normal(0,1.,N)),0,7)
    test= np.clip(55+7*apt+4*std+1.2*(slp-7)-1.5*sns+gke+R.normal(0,6,N),0,100)
    inc = R.lognormal(np.log(550),.45,N)
    df = pd.DataFrame({"学生ID":[f"26B{i+1:04d}" for i in range(N)],"学部":gk,"性別":gen,
        "身長cm":np.round(height,1),"一人暮らし":np.where(alone,"はい","いいえ"),
        "通学時間min":np.round(com).astype(int),"睡眠時間h":np.round(slp,1),"SNS時間h":np.round(sns,1),
        "勉強時間h":np.round(std,1),"アルバイト時間week":np.round(pt).astype(int),"出席率":np.round(att).astype(int),
        "朝食日数week":bf.astype(int),"テスト点":np.round(test).astype(int),"世帯年収万円":np.round(inc).astype(int)})
    df.loc[3,"世帯年収万円"]=12000; df.loc[88,"世帯年収万円"]=9500; df.loc[7,"身長cm"]=1710.0
    df.loc[15,"睡眠時間h"]=np.nan; df.loc[42,"睡眠時間h"]=np.nan; df.loc[101,"通学時間min"]=np.nan

print("学生数:", len(df))
df.head()

---
## フック：「平均的な北辰大生」は実在するか

ニュースは言う。「日本の世帯**平均**年収は◯◯万円」。

あなたは「自分の家、それより下かも…」と感じたことはないだろうか。実はその感覚、**正しい**かもしれない。なぜなら **平均は、ごく一部のお金持ちに簡単に吊り上げられる** からだ。

北辰大学の学生400人の `世帯年収万円` で確かめよう。

In [ ]:
inc = df["世帯年収万円"]
print(f"平均値　: {inc.mean():,.0f} 万円")
print(f"中央値　: {inc.median():,.0f} 万円   ← 400人を年収順に並べたとき真ん中の人")
print(f"最頻値　: {inc.mode().iloc[0]:,.0f} 万円付近   ← いちばん人数が多い帯")
print(f"最大値　: {inc.max():,.0f} 万円   ← この人が平均を吊り上げている")

**平均 ≒ 670万、中央値 ≒ 578万。** 100万円近くもズレている。

「平均年収670万」と聞くと豊かに見えるが、**実際の“真ん中の人”は578万**。平均は、たった数人の超富裕世帯に引っ張り上げられている。グラフで見よう。

In [ ]:
plt.figure(figsize=(8,4))
plt.hist(inc, bins=40, color="#80cbc4", edgecolor="white")
plt.axvline(inc.mean(),   color="#e8503a", lw=2, label=f"平均 {inc.mean():.0f}")
plt.axvline(inc.median(), color="#1565c0", lw=2, ls="--", label=f"中央値 {inc.median():.0f}")
plt.xlabel("世帯年収（万円）"); plt.ylabel("人数")
plt.title("右に裾を引く分布：平均が中央値より右へ引っ張られる")
plt.legend(); plt.show()

山は左側（300〜500万あたり）にあるのに、平均（赤線）は右にずれている。**分布が左右対称でないと、平均は山の位置を表さない。**

犯人は、はるか右にいる超富裕世帯（最大12,000万円）。これが**外れ値**だ。

---
## 外れ値は、平均をどれだけ動かすか

超富裕世帯（5,000万円超）を仮に除いてみる。平均と中央値が**それぞれどう動くか**に注目。

In [ ]:
正常 = inc[inc < 5000]   # 5000万円未満だけに絞る
print("          平均      中央値")
print(f"全員　  : {inc.mean():7.0f}   {inc.median():6.0f}")
print(f"外れ値除: {正常.mean():7.0f}   {正常.median():6.0f}")
print()
print(f"平均は {inc.mean()-正常.mean():.0f} 万円も動いた（外れ値に弱い）")
print(f"中央値はほぼ動かない（外れ値に強い＝頑健）")

たった数人を抜いただけで**平均は数十万円も動く**のに、**中央値はほとんど動かない**。中央値は外れ値に強い（これを「頑健 robust」と呼ぶ）。

> ⚠️ **では外れ値は消していいのか？**
>
> 答えは**「場合による」**。
> - 12,000万 → 入力ミスかもしれないし、本物の富裕層かもしれない。**ミスなら直す／除く。本物なら勝手に消すのは捏造**。
> - 「平均を下げたいから外れ値を消す」は禁じ手。外れ値の扱いは、**理由を説明できなければならない**。

---
## 対照実験：左右対称なデータなら、3つは一致する

年収は歪んでいたから3つがズレた。では**ほぼ左右対称**な `テスト点` ではどうか。

In [ ]:
t = df["テスト点"]
print(f"テスト点  平均 {t.mean():.1f} / 中央値 {t.median():.0f} / 最頻 {t.mode().iloc[0]:.0f}")
print("→ 3つがほぼ一致。対称な分布では『どれを使っても同じ』\n")

plt.figure(figsize=(8,3.5))
plt.hist(t, bins=25, color="#80cbc4", edgecolor="white")
plt.axvline(t.mean(), color="#e8503a", lw=2, label=f"平均 {t.mean():.1f}")
plt.axvline(t.median(), color="#1565c0", lw=2, ls="--", label=f"中央値 {t.median():.0f}")
plt.title("ほぼ対称な分布：平均と中央値がほぼ重なる")
plt.xlabel("テスト点"); plt.ylabel("人数"); plt.legend(); plt.show()

---
## 今日のまとめ

| 代表値 | 強み | 弱み | 向いている場面 |
|---|---|---|---|
| 平均値 | 全データを使う・計算しやすい | **外れ値・歪みに弱い** | 左右対称なデータ |
| 中央値 | 外れ値に強い（頑健） | 値の大小の情報を一部捨てる | **年収・地価など歪んだデータ** |
| 最頻値 | カテゴリにも使える | 連続データでは不安定 | アンケートの選択肢・人気 |

> **代表値の選択は、中立ではない。** 「平均年収」で語れば豊かに見え、「中央値」で語れば実感に近づく。
> どれを選ぶかは、すでに一つの**主張**である。だから――**数字を見たら、まず分布の形を疑え**。

**課題（Moodle）**：シナリオごとに「報告すべき代表値とその理由」を答える。